# Leakage audit of the original pipeline
Runtime > Run all. Reproduces the two checks used in the audit: (1) accuracy with and without the leaked columns (row index, label, action count), (2) whether adversarial files keep the row order of the original flows.

In [ ]:
import os, warnings
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")
try:
    from google.colab import drive
except ImportError:
    drive = None
if drive is not None:
    drive.mount("/content/drive")
BASE = os.environ.get("DRELAB_BASE", "/content/drive/MyDrive/ColabNotebooks/DReLAB")
OUT = os.environ.get("REV_OUT", "/content/drive/MyDrive/ColabNotebooks/revision_results")
os.makedirs(OUT, exist_ok=True)
FAM = {"CTU": ["Virut", "Neris", "Rbot", "Menti", "Murlo"], "BOTNET": ["Virut", "Neris", "Rbot", "Menti", "Murlo"], "CICIDS2018": ["Zeus_Ares"]}
ATT = [("RF", "DDQN"), ("RF", "SARSA"), ("WnD", "DDQN"), ("WnD", "SARSA")]


In [ ]:
# Leakage audit of the original pipeline (response to Reviewer 2, comment on leakage)
# Test 1: how much do the bookkeeping columns (Unnamed: 0, Label, n_actions) help a classifier?
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
DROP = ["Unnamed: 0", "Label", "n_actions"]
FIRST = {"CTU": "Virut", "BOTNET": "Neris", "CICIDS2018": "Zeus_Ares"}
def fit_acc(df, cols):
    X = df[cols].fillna(df[cols].median()); y = df["y"]
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, stratify=y, random_state=0)
    rf = RandomForestClassifier(100, n_jobs=-1, random_state=0).fit(Xtr, ytr); return round(rf.score(Xte, yte), 4), rf
def balanced(a, b, n=20000):
    n = min(n, len(a), len(b)); return pd.concat([a.sample(n, random_state=0).assign(y=0), b.sample(n, random_state=0).assign(y=1)], ignore_index=True)
print("=== TEST 1: accuracy with the leaked columns (notebook-style) and without them (clean features) ===")
for ds, fam in FIRST.items():
    ben = pd.read_csv(f"{BASE}/{ds}/benign.csv", nrows=60000); mal = pd.read_csv(f"{BASE}/{ds}/malicious/{fam}.csv"); adv = pd.read_csv(f"{BASE}/{ds}/RF/DDQN/{fam}.csv")
    for name, pos in [("benign vs original malware", mal), ("benign vs RF/DDQN adversarial", adv)]:
        df = balanced(ben, pos); paper_cols = [c for c in df.select_dtypes("number").columns if c != "y"]
        clean_cols = [c for c in ben.columns if c in pos.columns and c not in DROP]
        a_paper, rf = fit_acc(df, paper_cols); a_clean, _ = fit_acc(df, clean_cols)
        top = sorted(zip(rf.feature_importances_, paper_cols), reverse=True)[:3]
        print(f"{ds:10s} {name:31s} with leaked columns={a_paper}  clean features={a_clean}  top features={[(c, round(i, 2)) for i, c in top]}")
# Test 2: do adversarial files keep the row order of the originals? (low-cardinality columns match by chance, so this is only indicative)
print("\n=== TEST 2: same-position versus off-by-one match on categorical columns ===")
INV = ("IPSrcType", "IPDstType", "Proto", "sTos", "dTos", "SrcState", "DstState", "SrcPort", "DstPort", "->", ">", "<-", "<->")
for ds, fam in FIRST.items():
    mal = pd.read_csv(f"{BASE}/{ds}/malicious/{fam}.csv"); adv = pd.read_csv(f"{BASE}/{ds}/RF/DDQN/{fam}.csv")
    inv = [c for c in mal.columns if c in adv.columns and c.startswith(INV)]; m, a = mal[inv].values, adv[inv].values; L = min(len(m), len(a)); n = min(1000, L // 4)
    for start in [0, L // 2, L - n - 2]:
        mm, aa, a1 = m[start:start + n], a[start:start + n], a[start + 1:start + n + 1]
        k = min(len(mm), len(aa), len(a1))
        same = (mm[:k] == aa[:k]).all(axis=1).mean(); shifted = (mm[:k] == a1[:k]).all(axis=1).mean()
        print(f"{ds:10s} rows {start}-{start + n}: same-position match={same:.3f}  off-by-one match={shifted:.3f}")
